# Full Pipeline — Phase 3 + Phase 4 (Optimized)

**Phases 1 & 2 already completed.** This notebook runs Phase 3 (LoRA training) and Phase 4 (Robustness tests).

**Required data files (in Google Drive at `/MyDrive/nlp_econ/data/processed/`):**
- `labeled_chunk_registry.parquet`
- `labeled_document_registry.parquet`

**Run on Google Colab with GPU (A100 recommended, T4 works)**

**Outputs saved to:**
- Models: `/MyDrive/nlp_econ/models/`
- Results & plots: `/MyDrive/nlp_econ/outputs/`

**Notebook structure:**
- Cells 1-14: Phase 3 — LoRA Fine-Tuning (GPU required)
- Cells 15-22: Phase 4 — Robustness & Out-of-Sample (CPU-only)

---

## Changelog (vs. original Full_Pipeline.ipynb)

| # | Change | Why | Speedup |
|---|--------|-----|---------|
| 1 | **Removed Phase 1 & Phase 2 cells** | Already executed; data available as parquet files | N/A |
| 2 | **Removed `output_hidden_states=True`** — now uses `model.model.model()` to get `last_hidden_state` directly | Old code forced PyTorch to store all 32 intermediate layer hidden states (~500MB extra VRAM per forward pass), killing memory bandwidth | **2-3x** |
| 3 | **Added `torch.cuda.amp.autocast(dtype=bfloat16)`** around forward passes | A100 tensor cores are 2x faster in bf16 vs fp32; without autocast the regression head ran in fp32 | **1.5-2x** |
| 4 | **Pre-tokenization** — all texts tokenized once upfront in `__init__` | Original tokenized on-the-fly in `__getitem__` every batch, making GPU wait on CPU | **1.2-1.5x** |
| 5 | **DataLoader: `num_workers=2, pin_memory=True`** | Overlaps data loading with GPU compute; pinned memory enables async CPU→GPU transfer | **1.1-1.3x** |
| 6 | **Explicit `regression_head.to(device)`** | Original never moved the head to GPU — it silently ran on CPU causing slow CPU↔GPU data transfer | **correctness fix** |
| 7 | **Auto-detect GPU type** for optimal dtype | Uses `bfloat16` on A100/H100, falls back to `float16` on T4/V100 | optimal perf |
| 8 | **`bnb_4bit_compute_dtype` set to detected dtype** | Original hardcoded `float16` even on A100 where `bfloat16` is faster | **1.1x** |
| 9 | **Added `tqdm` progress bars** per batch | Original had zero output until epoch completed — no way to tell if running | visibility |
| 10 | **Added `non_blocking=True`** on `.to(device)` calls | Overlaps data transfer with computation | minor |
| 11 | **Added per-epoch timing** | Prints seconds per epoch so you can confirm speedup | visibility |
| 12 | **Added GPU memory diagnostics** | Prints VRAM usage after model load and peak usage after training | debugging |
| 13 | **TF32 enabled** (`torch.backends.cuda.matmul.allow_tf32 = True`) | Free ~10% speedup on Ampere+ GPUs for fp32 operations | **1.1x** |

**Combined expected improvement:** ~30 min/epoch → **5-8 min/epoch on A100**

---

In [ ]:
# Cell 1: Install ALL dependencies (Phase 3 + Phase 4)
!pip install -q transformers peft accelerate datasets bitsandbytes
!pip install -q scipy scikit-learn umap-learn
!pip install -q pyarrow fastparquet tqdm
!pip install -q seaborn matplotlib numpy pandas

### Cell 1b: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/nlp_econ')

# Phase 1-2 output files are in data/processed/
DATA_DIR = BASE_DIR / 'data' / 'processed'

# Save model checkpoints to models/
MODELS_DIR = BASE_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# Save all other outputs to outputs/
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Base dir: {BASE_DIR}')
print(f'Data dir: {DATA_DIR}')
print(f'Models dir: {MODELS_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print()
print('Files in data/processed/:')
for f in sorted(DATA_DIR.iterdir()):
    print(f'  {f.name}')


---
## Phase 3 — Advanced ML: LoRA Fine-Tuning
### Architecture: Mistral-7B-Instruct-v0.3 + LoRA + Linear Regression Head
---

### Cell 2: Imports & Config

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from scipy import stats
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import time

# Use the modern autocast API (torch 2.x compatible)
from torch.amp import autocast

# ============ CONFIG ============
SEED = 42
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_LENGTH = 512
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 8  # effective batch = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LABEL_COL = "delta_US_2Y_bp"
NUM_WORKERS = 2

# Temporal split (strict — no leakage)
TRAIN_END = "2018-12-31"
VAL_START = "2019-01-01"
VAL_END = "2020-12-31"
TEST_START = "2021-01-01"

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device & GPU optimizations
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    # A100/H100 use bfloat16; T4/V100 use float16
    USE_BF16 = "A100" in gpu_name or "A10" in gpu_name or "H100" in gpu_name
    COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    USE_BF16 = False
    COMPUTE_DTYPE = torch.float16

print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Compute dtype: {COMPUTE_DTYPE}")
print(f"Model: {MODEL_NAME}")
print(f"LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")


### Cell 3: Load Phase 1-2 Data

In [ ]:
# Load labeled chunks from Phase 1-2 output
chunks = pd.read_parquet(DATA_DIR / 'labeled_chunk_registry.parquet')
docs = pd.read_parquet(DATA_DIR / 'labeled_document_registry.parquet')

# Filter to Fed-only (US 2Y yield responds to Fed)
fed_docs = docs[docs["central_bank"] == "Board of Governors of the Federal Reserve"]
fed_doc_ids = set(fed_docs["doc_id"].values)

# Filter chunks
fed_chunks = chunks[chunks["doc_id"].isin(fed_doc_ids)].copy()
fed_chunks = fed_chunks[fed_chunks[LABEL_COL].notna()].copy()
fed_chunks["date"] = pd.to_datetime(fed_chunks["date"])

print(f"Fed chunks with labels: {len(fed_chunks)}")
print(f"Label stats: mean={fed_chunks[LABEL_COL].mean():.2f}, std={fed_chunks[LABEL_COL].std():.2f}")

# Temporal split
train = fed_chunks[fed_chunks["date"] <= TRAIN_END]
val = fed_chunks[(fed_chunks["date"] >= VAL_START) & (fed_chunks["date"] <= VAL_END)]
test = fed_chunks[fed_chunks["date"] >= TEST_START]

print(f"\nTemporal Split:")
print(f"  Train: {len(train)} chunks (up to {TRAIN_END})")
print(f"  Val:   {len(val)} chunks ({VAL_START} to {VAL_END})")
print(f"  Test:  {len(test)} chunks ({TEST_START} onward)")


### Cell 4: Pre-Tokenize Dataset (eliminates CPU bottleneck)

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


class PolicyStanceDataset(Dataset):
    """Pre-tokenized dataset for text -> yield change regression.
    
    Tokenizes ALL texts upfront so the DataLoader never waits on CPU.
    """
    
    def __init__(self, df, tokenizer, max_length=512, label_col="delta_US_2Y_bp"):
        self.labels = df[label_col].values.astype(np.float32)
        texts = [str(t)[:5000] for t in df["text"].tolist()]
        print(f"  Pre-tokenizing {len(texts)} samples...", end=" ")
        t0 = time.time()
        encodings = tokenizer(
            texts,
            max_length=max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        self.input_ids = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        print(f"done in {time.time()-t0:.1f}s | shape: {self.input_ids.shape}")
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }


print("Creating pre-tokenized datasets:")
train_dataset = PolicyStanceDataset(train, tokenizer, MAX_LENGTH, LABEL_COL)
val_dataset = PolicyStanceDataset(val, tokenizer, MAX_LENGTH, LABEL_COL)
test_dataset = PolicyStanceDataset(test, tokenizer, MAX_LENGTH, LABEL_COL)


### Cell 5: Load Model with 4-bit Quantization + LoRA

In [ ]:
# Quantization config — auto-selects optimal dtype for your GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Load base model
print(f"Loading {MODEL_NAME} with {COMPUTE_DTYPE} compute...")
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=COMPUTE_DTYPE,
)
base_model.config.use_cache = False
print(f"Loaded in {time.time()-t0:.1f}s")

# LoRA config
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

if device == 'cuda':
    print(f"\nGPU memory after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")


### Cell 6: Regression Head (optimized architecture)

In [ ]:
class StanceRegressionModel(nn.Module):
    """LLM + LoRA + Linear Regression Head for predicting delta_y2.
    
    OPTIMIZATION: Uses model.model.model (transformer body) directly
    instead of output_hidden_states=True. This avoids storing all 32
    intermediate layer hidden states in VRAM (saves ~500MB per forward pass).
    """
    
    def __init__(self, peft_model, hidden_size=4096):
        super().__init__()
        self.peft_model = peft_model
        self.regression_head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )
    
    def forward(self, input_ids, attention_mask, labels=None):
        # Access transformer body directly — no LM head, no stored intermediates
        outputs = self.peft_model.model.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_size)
        
        # Mean-pool over non-padding tokens
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        
        # Regression prediction (cast to float32 for stability)
        prediction = self.regression_head(pooled.float()).squeeze(-1)
        
        loss = None
        if labels is not None:
            loss = nn.MSELoss()(prediction, labels)
        
        return {"loss": loss, "predictions": prediction, "embeddings": pooled}


# Initialize
hidden_size = base_model.config.hidden_size
stance_model = StanceRegressionModel(model, hidden_size=hidden_size)

# CRITICAL: Move regression head to GPU
stance_model.regression_head = stance_model.regression_head.to(device)

print(f"Hidden size: {hidden_size}")
print(f"Regression head params: {sum(p.numel() for p in stance_model.regression_head.parameters()):,}")
print(f"Regression head on: {next(stance_model.regression_head.parameters()).device}")


### Cell 7: Training Setup

In [ ]:
# DataLoaders with workers + pin_memory for fast GPU feeding
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# Optimizer (only LoRA + regression head)
trainable_params = [p for p in stance_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)

# Cosine LR scheduler
total_steps = max(1, len(train_loader) * NUM_EPOCHS // GRADIENT_ACCUMULATION)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Batches/epoch: {len(train_loader)}")
print(f"Total optimizer steps: {total_steps}")


### Cell 8: Training Loop (with autocast + progress bars)

In [ ]:
def evaluate(model, dataloader, device):
    """Evaluate model with mixed precision."""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            
            with autocast("cuda", dtype=COMPUTE_DTYPE):
                outputs = model(input_ids, attention_mask, labels)
            
            total_loss += outputs["loss"].item()
            all_preds.extend(outputs["predictions"].float().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    preds = np.array(all_preds)
    labels_arr = np.array(all_labels)
    
    r = stats.pearsonr(preds, labels_arr)[0] if len(preds) > 2 else 0
    mae = mean_absolute_error(labels_arr, preds)
    dir_acc = np.mean(np.sign(preds) == np.sign(labels_arr))
    
    return {
        "loss": total_loss / len(dataloader),
        "pearson_r": r,
        "mae_bp": mae,
        "dir_accuracy": dir_acc,
    }


# === TRAINING LOOP ===
best_val_r = -1
train_losses = []
val_metrics_history = []

print("Starting training...")
print(f"{'Epoch':<7}{'Train Loss':<12}{'Val Loss':<10}{'Val r':<10}{'Val MAE':<10}{'Dir.Acc':<10}{'Time':<8}")
print("-" * 67)

total_train_start = time.time()

for epoch in range(NUM_EPOCHS):
    stance_model.train()
    epoch_loss = 0
    optimizer.zero_grad()
    epoch_start = time.time()
    
    progress_bar = tqdm(
        enumerate(train_loader), total=len(train_loader),
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False
    )
    
    for step, batch in progress_bar:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        
        # Mixed precision forward
        with autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = stance_model(input_ids, attention_mask, labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION
        
        loss.backward()
        epoch_loss += outputs["loss"].item()
        
        if (step + 1) % GRADIENT_ACCUMULATION == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        progress_bar.set_postfix({"loss": f"{epoch_loss/(step+1):.4f}"})
    
    # Handle remaining gradients if batches not divisible by GRADIENT_ACCUMULATION
    if (step + 1) % GRADIENT_ACCUMULATION != 0:
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    # Evaluate
    val_metrics = evaluate(stance_model, val_loader, device)
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    val_metrics_history.append(val_metrics)
    epoch_time = time.time() - epoch_start
    
    print(f"{epoch+1:<7}{avg_train_loss:<12.4f}{val_metrics['loss']:<10.4f}"
          f"{val_metrics['pearson_r']:<10.4f}{val_metrics['mae_bp']:<10.2f}"
          f"{val_metrics['dir_accuracy']:<10.3f}{epoch_time:<8.1f}s")
    
    # Save best checkpoint
    if val_metrics["pearson_r"] > best_val_r:
        best_val_r = val_metrics["pearson_r"]
        torch.save(stance_model.state_dict(), str(MODELS_DIR / 'best_stance_model.pt'))
        print(f"  -> New best model (r = {best_val_r:.4f})")
    
    # Early stopping (patience = 3)
    if epoch >= 3:
        recent = [m["pearson_r"] for m in val_metrics_history[-3:]]
        if all(val_r <= best_val_r - 0.01 for val_r in recent):
            print("Early stopping triggered.")
            break

total_time = time.time() - total_train_start
print(f"\nTraining complete in {total_time/60:.1f} minutes")
print(f"Best validation Pearson r: {best_val_r:.4f}")
if device == 'cuda':
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


### Cell 9: Final Test Evaluation

In [ ]:
# Load best model
stance_model.load_state_dict(torch.load(str(MODELS_DIR / 'best_stance_model.pt'), weights_only=True))
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

test_metrics = evaluate(stance_model, test_loader, device)
print("=" * 50)
print("FINAL TEST SET EVALUATION (2021-present)")
print("=" * 50)
print(f"  Pearson r:  {test_metrics['pearson_r']:.4f}")
print(f"  MAE (bp):   {test_metrics['mae_bp']:.2f}")
print(f"  Dir. Acc:   {test_metrics['dir_accuracy']:.3f}")
print(f"  Loss (MSE): {test_metrics['loss']:.4f}")
print("=" * 50)


### Cell 10: Embedding Extraction

In [ ]:
def extract_embeddings(model, dataloader, device):
    """Extract embeddings from the fine-tuned model."""
    model.eval()
    all_embeddings = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting embeddings"):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"]
            
            with autocast("cuda", dtype=COMPUTE_DTYPE):
                outputs = model(input_ids, attention_mask)
            
            embeddings = outputs["embeddings"].float().cpu().numpy()
            all_embeddings.append(embeddings)
            all_labels.extend(labels.numpy())
    
    return np.vstack(all_embeddings), np.array(all_labels)


print("Extracting embeddings from all Fed chunks...")
all_dataset = PolicyStanceDataset(fed_chunks, tokenizer, MAX_LENGTH, LABEL_COL)
all_loader = DataLoader(
    all_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

embeddings, labels = extract_embeddings(stance_model, all_loader, device)
print(f"Embedding matrix: {embeddings.shape}")
print(f"Labels: {labels.shape}")

np.save(str(OUTPUT_DIR / 'stance_embeddings.npy'), embeddings)
np.save(str(OUTPUT_DIR / 'stance_labels.npy'), labels)
print('Saved: stance_embeddings.npy, stance_labels.npy')


### Cell 11: PCA & Latent Stance Space

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# PCA fitted on TRAINING split only (no leakage!)
train_mask = fed_chunks["date"] <= TRAIN_END
train_emb = embeddings[train_mask.values]

# Standardize
scaler = StandardScaler()
train_emb_scaled = scaler.fit_transform(train_emb)

# PCA
pca = PCA(n_components=50)
pca.fit(train_emb_scaled)

# Transform all embeddings
all_emb_scaled = scaler.transform(embeddings)
all_pca = pca.transform(all_emb_scaled)

# PC1 = latent stance score
stance_scores = all_pca[:, 0]

# Correlation check
r_pc1, p_pc1 = stats.pearsonr(stance_scores[~np.isnan(labels)], 
                                labels[~np.isnan(labels)])
print(f"PC1 explained variance: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC1 correlation with delta_y2: r = {r_pc1:.4f}, p = {p_pc1:.4e}")
print(f"Top 5 PCs explain: {pca.explained_variance_ratio_[:5].sum()*100:.1f}%")

# Flip sign if needed for hawkish=positive
if r_pc1 < 0:
    stance_scores = -stance_scores
    r_pc1 = -r_pc1
    print("(Flipped PC1 sign for hawkish=positive convention)")

print(f"\nLatent stance score: r = {r_pc1:.4f} with delta_y2")
print("Positive = hawkish, Negative = dovish")


### Cell 12: UMAP Visualization

In [ ]:
import umap

# UMAP on PCA-reduced embeddings (50-dim -> 2-dim)
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, 
                     metric="cosine", random_state=SEED)
umap_emb = reducer.fit_transform(all_pca)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color by label
sc = axes[0].scatter(umap_emb[:, 0], umap_emb[:, 1], 
                      c=labels, cmap="RdBu_r", s=5, alpha=0.6,
                      vmin=-10, vmax=10)
axes[0].set_title("UMAP colored by delta_y2 (bp)")
plt.colorbar(sc, ax=axes[0], label="delta_y2 (bp)")

# Color by year
years = fed_chunks["date"].dt.year.values
sc2 = axes[1].scatter(umap_emb[:, 0], umap_emb[:, 1],
                       c=years, cmap="viridis", s=5, alpha=0.6)
axes[1].set_title("UMAP colored by Year")
plt.colorbar(sc2, ax=axes[1], label="Year")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'umap_stance_space.png'), dpi=150)
plt.show()
print("Latent stance space visualized.")


### Cell 13: Face Validity Check

In [ ]:
top_hawk_idx = np.argsort(stance_scores)[-10:][::-1]
top_dove_idx = np.argsort(stance_scores)[:10]

print("=" * 70)
print("TOP 10 MOST HAWKISH CHUNKS (by latent stance score)")
print("=" * 70)
for i, idx in enumerate(top_hawk_idx):
    text = fed_chunks.iloc[idx]["text"][:200]
    score = stance_scores[idx]
    print(f"\n[{i+1}] Score: {score:.3f}")
    print(f"    {text}...")

print("\n" + "=" * 70)
print("TOP 10 MOST DOVISH CHUNKS (by latent stance score)")
print("=" * 70)
for i, idx in enumerate(top_dove_idx):
    text = fed_chunks.iloc[idx]["text"][:200]
    score = stance_scores[idx]
    print(f"\n[{i+1}] Score: {score:.3f}")
    print(f"    {text}...")


### Cell 14: Save All Results

In [ ]:
# Save stance scores alongside chunk metadata
results_df = fed_chunks[["doc_id", "chunk_id", "date", "text"]].copy()
results_df["stance_score"] = stance_scores
results_df["delta_y2_bp"] = labels
results_df.to_parquet(str(OUTPUT_DIR / 'stance_scores_all_chunks.parquet'), index=False)

# Save training history
history = pd.DataFrame({
    "epoch": range(1, len(train_losses) + 1),
    "train_loss": train_losses,
    "val_loss": [m["loss"] for m in val_metrics_history],
    "val_pearson_r": [m["pearson_r"] for m in val_metrics_history],
    "val_mae_bp": [m["mae_bp"] for m in val_metrics_history],
    "val_dir_accuracy": [m["dir_accuracy"] for m in val_metrics_history],
})
history.to_csv(str(OUTPUT_DIR / 'training_history.csv'), index=False)

# Summary
print("=" * 70)
print("PHASE 3 COMPLETE")
print("=" * 70)
print("Files saved:")
print("  - best_stance_model.pt")
print("  - stance_embeddings.npy")
print("  - stance_labels.npy")
print("  - stance_scores_all_chunks.parquet")
print("  - training_history.csv")
print("  - umap_stance_space.png")
print()
print(f"  Best validation r: {best_val_r:.4f}")
print(f"  Test set r: {test_metrics['pearson_r']:.4f}")
print(f"  PC1 stance correlation: {r_pc1:.4f}")
print()
print("  Next: Proceed to Phase 4 below (Robustness)")


---
## Phase 4 — Robustness & Out-of-Sample Testing

**Milestones:**
- M7: Walk-Forward Cross-Validation & OOS Performance
- M8: Ablation Studies (isolating performance gains)
- M9: Economic Significance, Placebo Tests & Alternative Explanations

**Note:** Phase 4 is CPU-only (no GPU needed). Uses outputs from Phase 3.

---

### Cell 15: Phase 4 Setup

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit
import seaborn as sns

# Load stance scores (already in memory from Phase 3, but reload for safety)
scores_df = fed_chunks[["doc_id", "chunk_id", "date", "text"]].copy()
scores_df["stance_score"] = stance_scores
scores_df["delta_y2_bp"] = labels

print(f"Phase 4 working with {len(scores_df)} scored chunks")
print(f"Date range: {scores_df['date'].min().date()} to {scores_df['date'].max().date()}")


### Cell 16: M7 — Walk-Forward Cross-Validation

In [ ]:
# Aggregate chunk-level stance scores to event-level (document-day)
event_scores = scores_df.groupby(["doc_id", "date"]).agg(
    stance_mean=("stance_score", "mean"),
    stance_std=("stance_score", "std"),
    delta_y2=("delta_y2_bp", "first"),
    n_chunks=("stance_score", "count"),
).reset_index()
event_scores = event_scores.sort_values("date").reset_index(drop=True)

print(f"Event-level aggregation: {len(event_scores)} events")
print(f"Avg chunks per event: {event_scores['n_chunks'].mean():.1f}")

# Walk-forward validation (expanding window)
# Minimum 50 events for training, test on next 10
MIN_TRAIN = 50
TEST_WINDOW = 10

wf_results = []
n_events = len(event_scores)

for start in range(MIN_TRAIN, n_events - TEST_WINDOW, TEST_WINDOW):
    train_wf = event_scores.iloc[:start]
    test_wf = event_scores.iloc[start:start + TEST_WINDOW]
    
    # Simple linear: stance_mean -> delta_y2
    X_train = train_wf[["stance_mean"]].values
    y_train = train_wf["delta_y2"].values
    X_test = test_wf[["stance_mean"]].values
    y_test = test_wf["delta_y2"].values
    
    reg = LinearRegression().fit(X_train, y_train)
    preds = reg.predict(X_test)
    
    r_wf = stats.pearsonr(preds, y_test)[0] if len(preds) > 2 else 0
    mae_wf = mean_absolute_error(y_test, preds)
    dir_wf = np.mean(np.sign(preds) == np.sign(y_test))
    
    wf_results.append({
        "fold_start": test_wf["date"].iloc[0],
        "fold_end": test_wf["date"].iloc[-1],
        "n_train": len(train_wf),
        "pearson_r": r_wf,
        "mae_bp": mae_wf,
        "dir_accuracy": dir_wf,
    })

wf_df = pd.DataFrame(wf_results)
print("\n" + "=" * 60)
print("M7: WALK-FORWARD CROSS-VALIDATION RESULTS")
print("=" * 60)
print(f"  Folds: {len(wf_df)}")
print(f"  Mean Pearson r:    {wf_df['pearson_r'].mean():.4f} +/- {wf_df['pearson_r'].std():.4f}")
print(f"  Mean MAE (bp):     {wf_df['mae_bp'].mean():.2f}")
print(f"  Mean Dir. Acc:     {wf_df['dir_accuracy'].mean():.3f}")
print(f"  % Folds with r>0:  {(wf_df['pearson_r'] > 0).mean()*100:.0f}%")


### Cell 17: M7 — Walk-Forward Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pearson r over time
axes[0].plot(wf_df["fold_start"], wf_df["pearson_r"], marker="o", markersize=3)
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].axhline(wf_df["pearson_r"].mean(), color="green", linestyle="-", alpha=0.7, label=f"mean={wf_df['pearson_r'].mean():.3f}")
axes[0].set_title("Walk-Forward: Pearson r")
axes[0].set_xlabel("Fold Start Date")
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# Direction accuracy
axes[1].plot(wf_df["fold_start"], wf_df["dir_accuracy"], marker="o", markersize=3, color="orange")
axes[1].axhline(0.5, color="red", linestyle="--", alpha=0.5, label="random")
axes[1].set_title("Walk-Forward: Directional Accuracy")
axes[1].set_xlabel("Fold Start Date")
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

# Distribution of r values
axes[2].hist(wf_df["pearson_r"], bins=20, edgecolor="black", alpha=0.7)
axes[2].axvline(0, color="red", linestyle="--")
axes[2].set_title("Distribution of Fold r-values")
axes[2].set_xlabel("Pearson r")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'walk_forward_results.png'), dpi=150)
plt.show()


### Cell 18: M8 — Ablation Studies

In [ ]:
# Ablation: Compare different representations
# 1. Full model (LoRA fine-tuned stance score)
# 2. Random embeddings (control)
# 3. PC1 only vs multiple PCs

# Use test split for ablation
test_mask = fed_chunks["date"] >= TEST_START
test_emb = embeddings[test_mask.values]
test_labels = labels[test_mask.values]
test_scores = stance_scores[test_mask.values]

# Train split for fitting
train_mask_arr = (fed_chunks["date"] <= TRAIN_END).values
train_scores_abl = stance_scores[train_mask_arr]
train_labels_abl = labels[train_mask_arr]

ablation_results = {}

# --- Ablation 1: Full stance score (PC1) ---
reg_pc1 = LinearRegression().fit(train_scores_abl.reshape(-1, 1), train_labels_abl)
pred_pc1 = reg_pc1.predict(test_scores.reshape(-1, 1))
ablation_results["PC1 (Stance Score)"] = {
    "r": stats.pearsonr(pred_pc1, test_labels)[0],
    "mae": mean_absolute_error(test_labels, pred_pc1),
    "dir_acc": np.mean(np.sign(pred_pc1) == np.sign(test_labels)),
}

# --- Ablation 2: Top 5 PCs ---
train_pca5 = all_pca[train_mask_arr, :5]
test_pca5 = all_pca[test_mask.values, :5]
reg_pca5 = Ridge(alpha=1.0).fit(train_pca5, train_labels_abl)
pred_pca5 = reg_pca5.predict(test_pca5)
ablation_results["Top 5 PCs (Ridge)"] = {
    "r": stats.pearsonr(pred_pca5, test_labels)[0],
    "mae": mean_absolute_error(test_labels, pred_pca5),
    "dir_acc": np.mean(np.sign(pred_pca5) == np.sign(test_labels)),
}

# --- Ablation 3: Top 20 PCs ---
train_pca20 = all_pca[train_mask_arr, :20]
test_pca20 = all_pca[test_mask.values, :20]
reg_pca20 = Ridge(alpha=1.0).fit(train_pca20, train_labels_abl)
pred_pca20 = reg_pca20.predict(test_pca20)
ablation_results["Top 20 PCs (Ridge)"] = {
    "r": stats.pearsonr(pred_pca20, test_labels)[0],
    "mae": mean_absolute_error(test_labels, pred_pca20),
    "dir_acc": np.mean(np.sign(pred_pca20) == np.sign(test_labels)),
}

# --- Ablation 4: Random baseline ---
np.random.seed(SEED)
random_scores = np.random.randn(len(test_labels))
ablation_results["Random (control)"] = {
    "r": stats.pearsonr(random_scores, test_labels)[0],
    "mae": mean_absolute_error(test_labels, random_scores * test_labels.std()),
    "dir_acc": np.mean(np.sign(random_scores) == np.sign(test_labels)),
}

# Print ablation table
print("=" * 65)
print("M8: ABLATION STUDY — OOS Test Set (2021+)")
print("=" * 65)
print(f"{'Model':<25}{'Pearson r':<12}{'MAE (bp)':<12}{'Dir.Acc':<10}")
print("-" * 65)
for name, metrics in ablation_results.items():
    print(f"{name:<25}{metrics['r']:<12.4f}{metrics['mae']:<12.2f}{metrics['dir_acc']:<10.3f}")


### Cell 19: M9 — Placebo Tests

In [ ]:
# Placebo test: Shuffle labels and check if model still "works"
# If it does, the original result is likely spurious

N_PLACEBO = 1000
placebo_rs = []

for i in range(N_PLACEBO):
    shuffled_labels = np.random.permutation(test_labels)
    r_placebo = stats.pearsonr(test_scores, shuffled_labels)[0]
    placebo_rs.append(r_placebo)

placebo_rs = np.array(placebo_rs)
actual_r = stats.pearsonr(test_scores, test_labels)[0]

# p-value: fraction of placebo runs that beat actual
p_placebo = np.mean(np.abs(placebo_rs) >= np.abs(actual_r))

print("=" * 60)
print("M9: PLACEBO TEST (Label Permutation)")
print("=" * 60)
print(f"  Actual r (stance vs delta_y2):  {actual_r:.4f}")
print(f"  Placebo mean r:                 {placebo_rs.mean():.4f}")
print(f"  Placebo std r:                  {placebo_rs.std():.4f}")
print(f"  Placebo p-value:                {p_placebo:.4f}")
print(f"  Significance (p < 0.05):        {'YES' if p_placebo < 0.05 else 'NO'}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(placebo_rs, bins=50, alpha=0.7, label="Placebo distribution")
ax.axvline(actual_r, color="red", linewidth=2, label=f"Actual r = {actual_r:.4f}")
ax.axvline(-actual_r, color="red", linewidth=2, linestyle="--", alpha=0.5)
ax.set_xlabel("Pearson r")
ax.set_ylabel("Count")
ax.set_title(f"Placebo Test: p = {p_placebo:.4f} ({N_PLACEBO} permutations)")
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'placebo_test.png'), dpi=150)
plt.show()


### Cell 20: M9 — Economic Significance

In [ ]:
# Economic significance: Can the model generate a profitable trading signal?
# Simple strategy: go long/short based on predicted direction

# Event-level test set
test_events = event_scores[event_scores["date"] >= TEST_START].copy()

# Signal: sign of stance_mean (positive = hawkish = expect rate rise)
test_events["signal"] = np.sign(test_events["stance_mean"])
test_events["pnl_bp"] = test_events["signal"] * test_events["delta_y2"]

# Cumulative PnL
test_events["cum_pnl"] = test_events["pnl_bp"].cumsum()

# Metrics
total_pnl = test_events["pnl_bp"].sum()
sharpe = test_events["pnl_bp"].mean() / test_events["pnl_bp"].std() * np.sqrt(252) if test_events["pnl_bp"].std() > 0 else 0
hit_rate = (test_events["pnl_bp"] > 0).mean()
max_drawdown = (test_events["cum_pnl"].cummax() - test_events["cum_pnl"]).max()

print("=" * 60)
print("M9: ECONOMIC SIGNIFICANCE — Simple Direction Strategy")
print("=" * 60)
print(f"  Test period:    {test_events['date'].min().date()} to {test_events['date'].max().date()}")
print(f"  N events:       {len(test_events)}")
print(f"  Total PnL:      {total_pnl:.1f} bp")
print(f"  Annualized Sharpe: {sharpe:.2f}")
print(f"  Hit rate:       {hit_rate:.1%}")
print(f"  Max drawdown:   {max_drawdown:.1f} bp")

# Plot cumulative PnL
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test_events["date"], test_events["cum_pnl"], linewidth=1.5)
ax.axhline(0, color="black", linestyle="-", alpha=0.3)
ax.fill_between(test_events["date"], 0, test_events["cum_pnl"], alpha=0.2)
ax.set_title(f"Cumulative PnL (OOS) — Sharpe: {sharpe:.2f}")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative PnL (bp)")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'economic_significance.png'), dpi=150)
plt.show()


### Cell 21: M9 — Temporal Stability Check

In [ ]:
# Check if stance score correlation is stable across years
scores_df["year"] = scores_df["date"].dt.year

yearly_stats = []
for year, group in scores_df.groupby("year"):
    if len(group) > 10:
        r_year, p_year = stats.pearsonr(group["stance_score"], group["delta_y2_bp"])
        yearly_stats.append({
            "year": year,
            "n_chunks": len(group),
            "pearson_r": r_year,
            "p_value": p_year,
            "mean_stance": group["stance_score"].mean(),
        })

yearly_df = pd.DataFrame(yearly_stats)

print("=" * 60)
print("M9: TEMPORAL STABILITY — Yearly Correlation")
print("=" * 60)
print(f"{'Year':<8}{'N':<8}{'r':<10}{'p-value':<12}{'Mean Stance':<12}")
print("-" * 50)
for _, row in yearly_df.iterrows():
    sig = '*' if row['p_value'] < 0.05 else ' '
    print(f"{int(row['year']):<8}{int(row['n_chunks']):<8}{row['pearson_r']:<10.4f}{row['p_value']:<12.4e}{row['mean_stance']:<12.3f} {sig}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(yearly_df["year"], yearly_df["pearson_r"], color=["green" if r > 0 else "red" for r in yearly_df["pearson_r"]])
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_title("Yearly Stance-Yield Correlation")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Pearson r")

axes[1].plot(yearly_df["year"], yearly_df["mean_stance"], marker="o")
axes[1].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[1].set_title("Mean Stance Score by Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Mean Stance (+ = hawkish)")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'temporal_stability.png'), dpi=150)
plt.show()


### Cell 22: Final Summary & Saved Artifacts

In [ ]:
# Save Phase 4 results
wf_df.to_csv(str(OUTPUT_DIR / 'walk_forward_results.csv'), index=False)
yearly_df.to_csv(str(OUTPUT_DIR / 'yearly_stability.csv'), index=False)

print("=" * 70)
print("FULL PIPELINE COMPLETE (Phase 3 + Phase 4)")
print("=" * 70)
print()
print("Phase 3 outputs (in models/ and outputs/):")
print("  - models/best_stance_model.pt")
print("  - outputs/stance_embeddings.npy")
print("  - outputs/stance_labels.npy")
print("  - outputs/stance_scores_all_chunks.parquet")
print("  - outputs/training_history.csv")
print("  - outputs/umap_stance_space.png")
print()
print("Phase 4 outputs (in outputs/):")
print("  - outputs/walk_forward_results.csv")
print("  - outputs/walk_forward_results.png")
print("  - outputs/placebo_test.png")
print("  - outputs/economic_significance.png")
print("  - outputs/temporal_stability.png")
print("  - outputs/yearly_stability.csv")
print()
print("Key Results:")
print(f"  Phase 3 — Best val r:       {best_val_r:.4f}")
print(f"  Phase 3 — Test r:           {test_metrics['pearson_r']:.4f}")
print(f"  Phase 3 — PC1 stance r:     {r_pc1:.4f}")
print(f"  Phase 4 — Walk-forward r:   {wf_df['pearson_r'].mean():.4f} (mean)")
print(f"  Phase 4 — Placebo p-value:  {p_placebo:.4f}")
print(f"  Phase 4 — OOS Sharpe:       {sharpe:.2f}")
print()
print("Download all files for paper/thesis.")
